# 03 — Discrete-Time Engine

**Invariants proved:** Discrete-Time Engine + Branched Counterfactual Engine  
**Modules built:** `sdk/core/engine.py`, `sdk/core/scenario.py`

The engine manages the simulation loop, calling scenario hooks in the correct order and maintaining parallel factual/counterfactual trajectories. Three counterfactual modes:

| Mode | Behavior | Use case |
|------|----------|----------|
| `BRANCHED` | Full parallel trajectories | Default — captures compounding effects |
| `SNAPSHOT` | Same-step comparison only | V1 compatible, lower memory |
| `NONE` | Single trajectory | Performance, no counterfactuals needed |

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from sdk.core.engine import BranchedSimulationEngine, CounterfactualMode
from sdk.core.scenario import BaseScenario, TimeConfig, Predictions, Interventions, Outcomes

## Toy Counter Scenario

A trivial scenario that isolates engine behavior from domain complexity. Each entity has a counter that increments by 1 per step. `intervene()` adds 10 to all entities.

In [ ]:
class CounterScenario(BaseScenario[np.ndarray]):
    """State = array of counters. step adds 1, intervene adds 10."""
    unit_of_analysis = "counter"

    def create_population(self, n_entities: int) -> np.ndarray:
        return np.zeros(n_entities)

    def step(self, state: np.ndarray, t: int) -> np.ndarray:
        return state + 1

    def predict(self, state: np.ndarray, t: int) -> Predictions:
        return Predictions(scores=state.copy())

    def intervene(self, state, predictions, t):
        treated = predictions.scores > 0
        state = state.copy()
        state[treated] += 10
        return state, Interventions(treated_indices=np.where(treated)[0])

    def measure(self, state: np.ndarray, t: int) -> Outcomes:
        return Outcomes(events=state.copy(), entity_ids=np.arange(len(state)))

## 1. Hook Ordering

Verify the engine calls hooks in the correct order: `step -> predict -> intervene -> measure`.

In [ ]:
class LoggingScenario(BaseScenario[np.ndarray]):
    unit_of_analysis = "test"
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.log = []
    def create_population(self, n): self.log.append("create"); return np.zeros(n)
    def step(self, s, t): self.log.append(f"step({t})"); return s + 1
    def predict(self, s, t): self.log.append(f"predict({t})"); return Predictions(scores=s.copy())
    def intervene(self, s, p, t): self.log.append(f"intervene({t})"); return s, Interventions(treated_indices=np.array([]))
    def measure(self, s, t): self.log.append(f"measure({t})"); return Outcomes(events=s.copy())

tc = TimeConfig(n_timesteps=5, timestep_duration=1/52, prediction_schedule=[2])
sc = LoggingScenario(time_config=tc, seed=42)
BranchedSimulationEngine(sc, CounterfactualMode.NONE).run(2)

print("Call log:")
for entry in sc.log:
    print(f"  {entry}")

# Verify ordering at t=2
i_step = sc.log.index("step(2)")
i_pred = sc.log.index("predict(2)")
i_intv = sc.log.index("intervene(2)")
i_meas = sc.log.index("measure(2)")
assert i_step < i_pred < i_intv < i_meas
print("\nOrdering verified: step < predict < intervene < measure")

## 2. BRANCHED Mode: Factual vs Counterfactual Trajectories

The factual branch receives interventions (step + intervene), while the counterfactual evolves naturally (step only). They should diverge at intervention points.

In [ ]:
n_timesteps = 20
prediction_times = [5, 12]
tc = TimeConfig(n_timesteps=n_timesteps, timestep_duration=1/52, prediction_schedule=prediction_times)
sc = CounterScenario(time_config=tc, seed=42)
engine = BranchedSimulationEngine(sc, CounterfactualMode.BRANCHED)
results = engine.run(n_entities=3)

# Extract mean outcome at each timestep
factual = [results.outcomes[t].events.mean() for t in range(n_timesteps)]
counterfactual = [results.counterfactual_outcomes[t].events.mean() for t in range(n_timesteps)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(factual, 'b-o', markersize=4, label='Factual (with intervention)')
ax.plot(counterfactual, 'r--s', markersize=4, label='Counterfactual (no intervention)')
for pt in prediction_times:
    ax.axvline(x=pt, color='green', alpha=0.3, linestyle=':', linewidth=2)
    ax.annotate(f'intervene @ t={pt}', xy=(pt, factual[pt]), fontsize=8,
                xytext=(pt+0.5, factual[pt]+2), arrowprops=dict(arrowstyle='->', color='green'))
ax.set_xlabel('Timestep')
ax.set_ylabel('Mean counter value')
ax.set_title('BRANCHED Mode: Factual vs Counterfactual')
ax.legend()
plt.tight_layout()
plt.show()

# Verify expected values
print(f"Final factual mean:        {factual[-1]:.0f} (expected: {n_timesteps + len(prediction_times) * 10})")
print(f"Final counterfactual mean: {counterfactual[-1]:.0f} (expected: {n_timesteps})")
assert factual[-1] == n_timesteps + len(prediction_times) * 10
assert counterfactual[-1] == n_timesteps

## 3. All Three Modes Compared

Run the same scenario with all three counterfactual modes and compare results.

In [ ]:
modes = [CounterfactualMode.NONE, CounterfactualMode.SNAPSHOT, CounterfactualMode.BRANCHED]
tc = TimeConfig(n_timesteps=15, timestep_duration=1/52, prediction_schedule=[5, 10])

for mode in modes:
    sc = CounterScenario(time_config=tc, seed=42)
    results = BranchedSimulationEngine(sc, mode).run(3)
    n_factual = len(results.outcomes)
    n_cf = len(results.counterfactual_outcomes)
    final = results.outcomes[14].events.mean()
    print(f"{mode.value:10s}  factual_steps={n_factual:2d}  cf_steps={n_cf:2d}  final_mean={final:.0f}")

print("\nNONE: no counterfactuals. SNAPSHOT: CF only at prediction times. BRANCHED: CF at all times.")

## Key Insights

1. The engine enforces hook ordering: `step -> predict -> intervene -> measure` at each timestep.
2. `prediction_schedule` controls which timesteps trigger predict/intervene.
3. **BRANCHED** mode maintains full parallel trajectories — the factual and counterfactual evolve independently, diverging only at intervention points.
4. **SNAPSHOT** mode only captures same-step counterfactuals (cheaper but misses compounding effects).
5. All modes produce identical factual trajectories.

**Next:** [04_step_purity_contract.ipynb](04_step_purity_contract.ipynb) — the most important test in the SDK.